
# NanoVision — Optimization (Colab)

This notebook replicates the **Optimization** flow from:
- `src/pages/OptimizationPage.tsx`
- `src/pages/HistoryDetailPage.tsx`
- `src/lib/optimizer.ts`

### Included features
- History dataset management (upload existing history JSON or generate mock entries)
- Search and select optimization sample
- Original metrics panel
- **Run Risk Optimization** with visible process tracking/progress
- Optimized image generation (contrast/saturation/brightness enhancement)
- Optimized metrics panel (reduced risk)
- Editable optimized sample name
- Save optimized data back to selected entry + create snapshot history item
- Download optimized image
- Export updated history JSON


In [ ]:

#@title 1) Install dependencies (Colab)
!pip -q install ipywidgets pillow pandas matplotlib numpy


In [ ]:

#@title 2) Imports
import base64
import io
import json
import math
import random
import time
import uuid
from copy import deepcopy
from datetime import datetime

import ipywidgets as widgets
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from PIL import ImageEnhance
from IPython.display import HTML, Markdown, clear_output, display

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False


In [ ]:

#@title 3) Mock analysis generator (compatible with Optimization fields)
def clamp(value, min_value=0, max_value=100):
    return max(min_value, min(max_value, value))

MOCK_COMPOUNDS = [
    {"smiles": "CC(=O)OC1=CC=CC=C1C(=O)O", "molecularWeight": 180.16, "solubility": 3.2},
    {"smiles": "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O", "molecularWeight": 206.28, "solubility": 0.08},
    {"smiles": "CC(=O)NC1=CC=C(O)C=C1", "molecularWeight": 151.16, "solubility": 14.0},
    {"smiles": "CN1CCC[C@H]1C2=CN=CC=C2", "molecularWeight": 162.23, "solubility": 10.5},
    {"smiles": "C1=CC=C(C=C1)C=O", "molecularWeight": 106.12, "solubility": 6.7},
]

def run_mock_analysis(seed=None):
    if seed is not None:
        random.seed(seed)

    nuclei_count = random.randint(30, 149)
    mean_area = round(random.random() * 500 + 200, 1)
    std_area = round(random.random() * 100 + 20, 1)
    circularity = round(random.random() * 0.4 + 0.6, 3)
    aggregation_score = round(random.random() * 0.6 + 0.1, 2)
    dice_score = round(random.random() * 0.15 + 0.82, 3)
    iou_score = round(dice_score - random.random() * 0.1, 3)
    density_per_unit = round(nuclei_count / (random.random() * 5 + 8), 1)
    stability_score = round(random.random() * 40 + 55, 1)
    uniformity_score = round(random.random() * 35 + 60, 1)
    interaction_strength = round(random.random() * 50 + 40, 1)

    total = stability_score + uniformity_score + (100 - aggregation_score * 100) + interaction_strength
    screening_decision = "Promising Candidate" if total > 300 else "Needs Optimization" if total > 220 else "Low Performance"

    density_data = [
        {"region": "Q1", "density": random.randint(5, 34)},
        {"region": "Q2", "density": random.randint(5, 34)},
        {"region": "Q3", "density": random.randint(5, 34)},
        {"region": "Q4", "density": random.randint(5, 34)},
    ]

    dynamics_data = []
    for i, t in enumerate(["T0", "T1", "T2", "T3", "T4", "T5"]):
        dynamics_data.append({
            "t": t,
            "diffusion": round(clamp(42 + i * 8 + random.random() * 10), 1),
            "movement": round(clamp(34 + i * 9 + random.random() * 9), 1),
            "cellResponse": round(clamp(38 + i * 7 + random.random() * 8), 1),
        })

    risk_score = clamp(100 - (stability_score * 0.35 + uniformity_score * 0.25 + interaction_strength * 0.2 + (100 - aggregation_score * 100) * 0.2))
    multi_factor_score = clamp(stability_score * 0.4 + uniformity_score * 0.3 + interaction_strength * 0.3)
    area_comparison_score = clamp(100 - std_area * 0.7)
    aggregation_detection_score = clamp(aggregation_score * 100)
    psnr = round(22 + random.random() * 15, 2)
    ssim = round(0.75 + random.random() * 0.23, 3)
    structural_clarity = clamp((psnr - 20) * 5)
    segmentation_confidence = clamp((dice_score * 100 + iou_score * 100) / 2)
    membrane_interaction_score = clamp(interaction_strength)
    cytotoxicity_risk = clamp(aggregation_score * 90 + (1 - circularity) * 25)
    surface_stability = clamp(stability_score - aggregation_score * 15)
    zeta_potential_proxy = round(-35 + random.random() * 45, 1)
    diffusion_coefficient = round(0.15 + random.random() * 0.9, 3)
    transport_efficiency = clamp(uniformity_score * 0.6 + stability_score * 0.4)
    bioavailability_prediction = clamp(transport_efficiency * 0.65 + interaction_strength * 0.35)
    structural_consistency = clamp(100 - std_area * 0.5 + circularity * 15)
    cluster_formation = clamp(aggregation_score * 100)
    density_variation = clamp(abs(density_data[0]['density'] - density_data[2]['density']) * 2.1)
    particle_overlap = clamp(aggregation_score * 80 + (100 - circularity * 100) * 0.25)
    stability_risk = clamp((100 - stability_score) * 0.6 + cluster_formation * 0.4)
    feature_vector_integration = clamp((segmentation_confidence + membrane_interaction_score + bioavailability_prediction) / 3)
    weighted_score = clamp(feature_vector_integration * 0.45 + (100 - risk_score) * 0.55)
    threshold_gap = round(weighted_score - 62, 2)
    final_screening_score = clamp(weighted_score - risk_score * 0.2)
    model_head_risk = clamp(100 / (1 + math.exp(-(risk_score - 45) / 8)))

    selected_compound = random.choice(MOCK_COMPOUNDS)
    smiles = selected_compound['smiles']
    molecular_weight = round(selected_compound['molecularWeight'] + (random.random() * 6 - 3), 2)
    binding_affinity = round(-5.2 - random.random() * 6.4, 2)
    solubility = round(clamp(selected_compound['solubility'] + (random.random() * 2.4 - 1.2), 0.02, 20), 2)
    cell_uptake_rate = round(clamp(45 + random.random() * 45), 1)
    toxicity_label = 'Low' if cytotoxicity_risk < 35 else 'Moderate' if cytotoxicity_risk < 62 else 'High'
    protein_interaction = round(clamp(interaction_strength + random.random() * 10), 1)
    target_receptor_binding = round(clamp(55 + random.random() * 35), 1)
    latest = dynamics_data[-1]
    diffusion_trend = round(latest['diffusion'], 1)
    movement_trend = round(latest['movement'], 1)
    response_trend = round(latest['cellResponse'], 1)
    formulation_transport_score = round(clamp((transport_efficiency + diffusion_trend) / 2), 1)
    predicted_drug_efficacy = round(clamp((bioavailability_prediction * 0.35) + (target_receptor_binding * 0.25) + (protein_interaction * 0.2) + (100 - cytotoxicity_risk) * 0.2), 1)
    predictive_toxicity_score = round(clamp(cytotoxicity_risk * 0.7 + model_head_risk * 0.3), 1)
    multi_criteria_screening_score = round(clamp(predicted_drug_efficacy * 0.65 + (100 - predictive_toxicity_score) * 0.35), 1)
    synthesis_yield = round(clamp(weighted_score + (random.random() * 8 - 4)), 1)
    pharmacodynamics_index = round(clamp((predicted_drug_efficacy + protein_interaction + target_receptor_binding) / 3), 1)
    automated_decision = 'Promising Candidate' if multi_criteria_screening_score >= 72 else 'Needs Optimization' if multi_criteria_screening_score >= 52 else 'Reject'

    return {
        'nucleiCount': nuclei_count,
        'meanArea': mean_area,
        'stdArea': std_area,
        'circularity': circularity,
        'aggregationScore': aggregation_score,
        'diceScore': dice_score,
        'iouScore': iou_score,
        'densityPerUnit': density_per_unit,
        'stabilityScore': stability_score,
        'uniformityScore': uniformity_score,
        'interactionStrength': interaction_strength,
        'screeningDecision': screening_decision,
        'particleSizes': [],
        'densityData': density_data,
        'radarData': [],
        'dynamicsData': dynamics_data,
        'screeningMetrics': {
            'riskScore': risk_score,
            'multiFactorScore': multi_factor_score,
            'areaComparisonScore': area_comparison_score,
            'aggregationDetectionScore': aggregation_detection_score,
            'psnr': psnr,
            'ssim': ssim,
            'structuralClarity': structural_clarity,
            'segmentationConfidence': segmentation_confidence,
            'membraneInteractionScore': membrane_interaction_score,
            'cytotoxicityRisk': cytotoxicity_risk,
            'surfaceStability': surface_stability,
            'zetaPotentialProxy': zeta_potential_proxy,
            'diffusionCoefficient': diffusion_coefficient,
            'transportEfficiency': transport_efficiency,
            'bioavailabilityPrediction': bioavailability_prediction,
            'structuralConsistency': structural_consistency,
            'clusterFormation': cluster_formation,
            'densityVariation': density_variation,
            'particleOverlap': particle_overlap,
            'stabilityRisk': stability_risk,
            'featureVectorIntegration': feature_vector_integration,
            'weightedScore': weighted_score,
            'thresholdGap': threshold_gap,
            'finalScreeningScore': final_screening_score,
            'modelHeadRisk': model_head_risk,
            'smiles': smiles,
            'molecularWeight': molecular_weight,
            'bindingAffinity': binding_affinity,
            'solubility': solubility,
            'cellUptakeRate': cell_uptake_rate,
            'toxicityLabel': toxicity_label,
            'proteinInteraction': protein_interaction,
            'targetReceptorBinding': target_receptor_binding,
            'diffusionTrend': diffusion_trend,
            'movementTrend': movement_trend,
            'responseTrend': response_trend,
            'formulationTransportScore': formulation_transport_score,
            'predictedDrugEfficacy': predicted_drug_efficacy,
            'predictiveToxicityScore': predictive_toxicity_score,
            'multiCriteriaScreeningScore': multi_criteria_screening_score,
            'dockingAffinity': binding_affinity,
            'synthesisYield': synthesis_yield,
            'pharmacodynamicsIndex': pharmacodynamics_index,
            'automatedDecision': automated_decision,
        },
    }


In [ ]:

#@title 4) Optimizer functions (mirrors src/lib/optimizer.ts)
def optimize_for_lower_risk(result):
    optimized = deepcopy(result)

    optimized['aggregationScore'] = round(max(0.05, result['aggregationScore'] * 0.72), 2)
    optimized['circularity'] = round(min(0.99, result['circularity'] + 0.06), 3)
    optimized['stabilityScore'] = round(clamp(result['stabilityScore'] + 9), 1)
    optimized['uniformityScore'] = round(clamp(result['uniformityScore'] + 7), 1)
    optimized['interactionStrength'] = round(clamp(result['interactionStrength'] + 4), 1)

    m = optimized['screeningMetrics']
    rm = result['screeningMetrics']
    m['riskScore'] = round(clamp(rm['riskScore'] * 0.72), 1)
    m['cytotoxicityRisk'] = round(clamp(rm['cytotoxicityRisk'] * 0.8), 1)
    m['clusterFormation'] = round(clamp(rm['clusterFormation'] * 0.7), 1)
    m['particleOverlap'] = round(clamp(rm['particleOverlap'] * 0.75), 1)
    m['stabilityRisk'] = round(clamp(rm['stabilityRisk'] * 0.72), 1)
    m['surfaceStability'] = round(clamp(rm['surfaceStability'] + 7), 1)
    m['transportEfficiency'] = round(clamp(rm['transportEfficiency'] + 8), 1)
    m['bioavailabilityPrediction'] = round(clamp(rm['bioavailabilityPrediction'] + 7), 1)
    m['featureVectorIntegration'] = round(clamp(rm['featureVectorIntegration'] + 6), 1)
    m['weightedScore'] = round(clamp(rm['weightedScore'] + 6), 1)
    m['finalScreeningScore'] = round(clamp(rm['finalScreeningScore'] + 11), 1)
    m['thresholdGap'] = round(m['weightedScore'] - 62, 2)
    m['modelHeadRisk'] = round(clamp(rm['modelHeadRisk'] * 0.76), 1)
    m['cellUptakeRate'] = round(clamp(rm['cellUptakeRate'] + 5), 1)
    m['proteinInteraction'] = round(clamp(rm['proteinInteraction'] + 4), 1)
    m['targetReceptorBinding'] = round(clamp(rm['targetReceptorBinding'] + 5), 1)
    m['formulationTransportScore'] = round(clamp(rm['formulationTransportScore'] + 6), 1)
    m['predictedDrugEfficacy'] = round(clamp(rm['predictedDrugEfficacy'] + 8), 1)
    m['predictiveToxicityScore'] = round(clamp(rm['predictiveToxicityScore'] * 0.8), 1)
    m['multiCriteriaScreeningScore'] = round(clamp(rm['multiCriteriaScreeningScore'] + 8), 1)
    m['synthesisYield'] = round(clamp(rm['synthesisYield'] + 6), 1)
    m['pharmacodynamicsIndex'] = round(clamp(rm['pharmacodynamicsIndex'] + 5), 1)
    m['automatedDecision'] = 'Promising Candidate' if m['multiCriteriaScreeningScore'] >= 72 else 'Needs Optimization' if m['multiCriteriaScreeningScore'] >= 52 else 'Reject'
    m['toxicityLabel'] = 'Low' if m['predictiveToxicityScore'] < 35 else 'Moderate' if m['predictiveToxicityScore'] < 62 else 'High'

    optimized['screeningDecision'] = 'Promising Candidate' if m['finalScreeningScore'] >= 80 else 'Needs Optimization' if m['finalScreeningScore'] >= 62 else 'Low Performance'
    return optimized


def pil_image_to_data_url(img, fmt='PNG'):
    buffer = io.BytesIO()
    img.save(buffer, format=fmt)
    encoded = base64.b64encode(buffer.getvalue()).decode('utf-8')
    return f"data:image/{fmt.lower()};base64,{encoded}"


def data_url_to_pil(data_url):
    if not data_url or ',' not in data_url:
        return None
    b64 = data_url.split(',', 1)[1]
    data = base64.b64decode(b64)
    return PILImage.open(io.BytesIO(data))


def create_optimized_image_data(image_data):
    img = data_url_to_pil(image_data)
    if img is None:
        return image_data

    img = img.convert('RGB')
    img = ImageEnhance.Contrast(img).enhance(1.08)
    img = ImageEnhance.Color(img).enhance(1.05)
    img = ImageEnhance.Brightness(img).enhance(1.02)
    return pil_image_to_data_url(img, fmt='PNG')


def default_mock_image_data(seed=0, w=480, h=320):
    rng = np.random.default_rng(seed)
    arr = rng.normal(loc=120, scale=35, size=(h, w, 3)).clip(0, 255).astype(np.uint8)
    img = PILImage.fromarray(arr, mode='RGB')
    return pil_image_to_data_url(img, fmt='PNG')


In [ ]:

#@title 5) History helpers

def make_history_entry(name, seed=0):
    return {
        'id': str(uuid.uuid4()),
        'createdAt': datetime.utcnow().isoformat() + 'Z',
        'imageName': name,
        'imageData': default_mock_image_data(seed=seed),
        'result': run_mock_analysis(seed=seed),
        'optimizedResult': None,
        'optimizedImageData': None,
        'optimizedName': None,
    }

state = {
    'history': [],
    'selected_id': None,
    'draft_result': None,
    'draft_image': None,
    'draft_name': '',
    'save_message': '',
}

def get_entry_by_id(entry_id):
    for e in state['history']:
        if e['id'] == entry_id:
            return e
    return None


def update_entry_by_id(entry_id, updater):
    for i, e in enumerate(state['history']):
        if e['id'] == entry_id:
            state['history'][i] = updater(e)
            return state['history'][i]
    return None


In [ ]:

#@title 6) Optimization dashboard UI
history_upload = widgets.FileUpload(accept='.json', multiple=False, description='Upload History')
mock_count = widgets.IntSlider(value=5, min=1, max=20, step=1, description='Mock count')
btn_gen = widgets.Button(description='Generate Mock History', button_style='info', icon='random')
btn_export = widgets.Button(description='Export History JSON', button_style='success', icon='download')

search_input = widgets.Text(description='Search', placeholder='Search optimization sample')
entry_select = widgets.Dropdown(options=[], description='Open Sample')
status = widgets.HTML('<b>Status:</b> Ready.')

btn_run_opt = widgets.Button(description='Run Risk Optimization', button_style='primary', icon='magic')
progress_bar = widgets.IntProgress(value=0, min=0, max=100, description='Progress')
name_input = widgets.Text(description='Save Name', placeholder='optimized sample name')
btn_save = widgets.Button(description='Save Optimized Data', button_style='warning', icon='save')
btn_download = widgets.Button(description='Download Optimized Image', button_style='success', icon='download')

out_history_table = widgets.Output()
out_original = widgets.Output()
out_optimized = widgets.Output()


def refresh_entry_options():
    q = search_input.value.strip().lower()
    filtered = [e for e in state['history'] if q in e['imageName'].lower()]
    opts = [(f"{e['imageName']} | {e['createdAt'][:19]}", e['id']) for e in filtered]
    if not opts:
        entry_select.options = [('-- no match --', None)]
        entry_select.value = None
        state['selected_id'] = None
        return

    entry_select.options = opts
    valid_ids = {v for _, v in opts}
    if state['selected_id'] not in valid_ids:
        state['selected_id'] = opts[0][1]
    entry_select.value = state['selected_id']


def render_history_table():
    with out_history_table:
        clear_output(wait=True)
        if not state['history']:
            display(Markdown('No history entries yet. Generate mock or upload JSON.'))
            return
        rows = []
        for e in state['history']:
            rows.append({
                'Image': e['imageName'],
                'Original risk': round(e['result']['screeningMetrics']['riskScore'], 1),
                'Optimized risk': '-' if not e.get('optimizedResult') else round(e['optimizedResult']['screeningMetrics']['riskScore'], 1),
                'Saved as': e.get('optimizedName') or (e['imageName'].rsplit('.',1)[0] + '-optimized' if e.get('optimizedResult') else 'not saved'),
                'Record time': e['createdAt'][:19],
            })
        display(pd.DataFrame(rows))


def render_selected_panels():
    entry = get_entry_by_id(state['selected_id'])
    with out_original:
        clear_output(wait=True)
        if not entry:
            display(Markdown('No sample selected.'))
            return
        m = entry['result']['screeningMetrics']
        display(Markdown(f"### Original Results — `{entry['imageName']}`"))
        display(Markdown(f"- Decision: **{entry['result']['screeningDecision']}**
- Risk Score: **{m['riskScore']:.1f}**
- Final Score: **{m['finalScreeningScore']:.1f}**
- Stability Risk: **{m['stabilityRisk']:.1f}**"))
        img = data_url_to_pil(entry['imageData'])
        if img is not None:
            display(img.resize((min(640, img.width), int(img.height * min(640, img.width) / img.width))))

    with out_optimized:
        clear_output(wait=True)
        draft = state['draft_result']
        if not draft:
            display(Markdown('### Optimized Results
Run optimization first.'))
            return
        m = draft['screeningMetrics']
        display(Markdown('### Optimized Results (Reduced Risk)'))
        display(Markdown(f"- Decision: **{draft['screeningDecision']}**
- Risk Score: **{m['riskScore']:.1f}**
- Final Score: **{m['finalScreeningScore']:.1f}**
- Stability Risk: **{m['stabilityRisk']:.1f}**"))
        img = data_url_to_pil(state['draft_image']) if state['draft_image'] else None
        if img is not None:
            display(img.resize((min(640, img.width), int(img.height * min(640, img.width) / img.width))))


def load_selected_entry_draft():
    entry = get_entry_by_id(state['selected_id'])
    if not entry:
        state['draft_result'] = None
        state['draft_image'] = None
        state['draft_name'] = ''
        name_input.value = ''
        return
    state['draft_result'] = entry.get('optimizedResult')
    state['draft_image'] = entry.get('optimizedImageData')
    default_name = f"{entry['imageName'].rsplit('.',1)[0]}-optimized"
    state['draft_name'] = entry.get('optimizedName') or default_name
    name_input.value = state['draft_name']


def rerender_all():
    refresh_entry_options()
    render_history_table()
    load_selected_entry_draft()
    render_selected_panels()


def on_generate(_):
    n = int(mock_count.value)
    state['history'] = [make_history_entry(f'sample_{i+1:03d}.png', seed=100+i) for i in range(n)]
    state['selected_id'] = state['history'][0]['id'] if state['history'] else None
    status.value = f"<b>Status:</b> Generated {n} mock entries for optimization."
    rerender_all()


def on_upload(change):
    if not history_upload.value:
        return
    file_info = next(iter(history_upload.value.values()))
    try:
        parsed = json.loads(file_info['content'].decode('utf-8'))
        if not isinstance(parsed, list):
            raise ValueError('History JSON must be a list.')
        state['history'] = parsed
        state['selected_id'] = parsed[0]['id'] if parsed else None
        status.value = f"<b>Status:</b> Loaded {len(parsed)} history entries."
    except Exception as e:
        status.value = f"<b>Status:</b> Failed to parse upload: {e}"
    rerender_all()


def on_export(_):
    payload = json.dumps(state['history'], indent=2)
    with open('nanovision_history_optimized.json', 'w', encoding='utf-8') as f:
        f.write(payload)
    status.value = '<b>Status:</b> Exported nanovision_history_optimized.json.'
    if IN_COLAB:
        files.download('nanovision_history_optimized.json')


def on_search(change):
    refresh_entry_options()


def on_select(change):
    state['selected_id'] = entry_select.value
    load_selected_entry_draft()
    render_selected_panels()


def on_name_change(change):
    state['draft_name'] = name_input.value


def on_run_opt(_):
    entry = get_entry_by_id(state['selected_id'])
    if not entry:
        status.value = '<b>Status:</b> Select a history sample first.'
        return

    status.value = '<b>Status:</b> Optimizing sample pipeline...'
    progress_bar.value = 0
    for p in [8, 16, 24, 32, 40, 48, 56, 64, 72, 80, 88, 92]:
        progress_bar.value = p
        time.sleep(0.08)

    state['draft_result'] = optimize_for_lower_risk(entry['result'])
    try:
        state['draft_image'] = create_optimized_image_data(entry['imageData'])
    except Exception:
        state['draft_image'] = entry['imageData']

    if not state['draft_name']:
        state['draft_name'] = entry.get('optimizedName') or f"{entry['imageName'].rsplit('.',1)[0]}-optimized"
        name_input.value = state['draft_name']

    progress_bar.value = 100
    status.value = '<b>Status:</b> Optimization completed.'
    render_selected_panels()


def on_save(_):
    entry = get_entry_by_id(state['selected_id'])
    if not entry:
        status.value = '<b>Status:</b> Select a sample first.'
        return

    result_to_save = state['draft_result'] or entry.get('optimizedResult')
    image_to_save = state['draft_image'] or entry.get('optimizedImageData') or entry.get('imageData')
    name_to_save = state['draft_name'] or entry.get('optimizedName') or f"{entry['imageName'].rsplit('.',1)[0]}-optimized"

    if not result_to_save:
        status.value = '<b>Status:</b> Run optimization first before saving.'
        return

    snapshot_image_name = name_to_save if '.' in name_to_save else f"{name_to_save}.png"

    updated = update_entry_by_id(entry['id'], lambda current: {
        **current,
        'optimizedResult': result_to_save,
        'optimizedImageData': image_to_save,
        'optimizedName': name_to_save,
    })

    if not updated:
        status.value = '<b>Status:</b> Save failed: selected entry not found.'
        return

    snapshot = {
        'id': str(uuid.uuid4()),
        'createdAt': datetime.utcnow().isoformat() + 'Z',
        'imageName': snapshot_image_name,
        'imageData': image_to_save,
        'result': result_to_save,
        'optimizedResult': result_to_save,
        'optimizedImageData': image_to_save,
        'optimizedName': name_to_save,
    }
    state['history'] = [snapshot] + state['history']

    status.value = f"<b>Status:</b> Optimized data saved successfully. Snapshot ID: {snapshot['id']}"
    render_history_table()


def on_download(_):
    entry = get_entry_by_id(state['selected_id'])
    if not entry:
        status.value = '<b>Status:</b> Select a sample first.'
        return

    data_url = state['draft_image'] or entry.get('optimizedImageData') or entry.get('imageData')
    name = state['draft_name'] or entry.get('optimizedName') or entry['imageName']
    img = data_url_to_pil(data_url)
    if img is None:
        status.value = '<b>Status:</b> Download failed: image data unavailable.'
        return

    file_name = f"{name if '.' in name else name + '.png'}"
    img.save(file_name)
    status.value = f"<b>Status:</b> Download-ready file written: {file_name}"
    if IN_COLAB:
        files.download(file_name)


btn_gen.on_click(on_generate)
btn_export.on_click(on_export)
history_upload.observe(on_upload, names='value')
search_input.observe(on_search, names='value')
entry_select.observe(on_select, names='value')
name_input.observe(on_name_change, names='value')
btn_run_opt.on_click(on_run_opt)
btn_save.on_click(on_save)
btn_download.on_click(on_download)


In [ ]:

#@title 7) Launch Optimization app
ui = widgets.VBox([
    widgets.HTML('<h2>Optimization</h2><p>Open any history sample and run optimization with process tracking, editable save name, and image download.</p>'),
    widgets.HBox([history_upload, mock_count, btn_gen, btn_export]),
    status,
    widgets.HBox([search_input, entry_select]),
    out_history_table,
    widgets.HTML('<hr><h4>Run Optimizer</h4>'),
    btn_run_opt,
    progress_bar,
    widgets.HTML('<hr><h4>Original vs Optimized</h4>'),
    out_original,
    out_optimized,
    name_input,
    widgets.HBox([btn_save, btn_download]),
])

display(ui)
rerender_all()
